# Cruise Ship Data Cleaning
This notebook documents the cleaning and preprocessing of cruise ship arrival data for Dubrovnik during 2024 and 2025, preparing the dataset for integration with Uber trip records.

## Data loading and dataset overview

In [32]:
import pandas as pd
import numpy as np

In [33]:
df = pd.read_csv('../../data/raw_data/ship_hourly_2024_2025.csv')

In [34]:
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,CRUISE SHIP ARRIVALS TO DUBROVNIK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Berth,NaN,2024,NaN,NaN
1,Time of IMO number Ship's name Ship's flag Name of the Agent Date & time of ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Gruž,Old\nTown,Len(m),GT,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,401,119,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,0,NaN,NaN,NaN
4,03.01.2024,NaN,NaN,Wednesday,NaN,NaN,NaN,NaN,NaN,1,0,NaN,NaN,NaN


In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1636 entries, 0 to 1635
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       1524 non-null   object
 1   1       950 non-null    object
 2   2       946 non-null    object
 3   3       1414 non-null   object
 4   4       947 non-null    object
 5   5       948 non-null    object
 6   6       948 non-null    object
 7   7       945 non-null    object
 8   8       948 non-null    object
 9   9       1631 non-null   object
 10  10      645 non-null    object
 11  11      1055 non-null   object
 12  12      1000 non-null   object
 13  13      16 non-null     object
dtypes: object(14)
memory usage: 179.1+ KB


In [36]:
df.columns.tolist()

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13']

In [37]:
df.iloc[5:25, 0:3]

,0,1,2
5,21:25,9398008,ATHENA
6,04.01.2024,NaN,NaN
7,08:12,9398917,COSTA\nDELIZIOSA
8,05.01.2024,NaN,NaN
9,00:13,9398008,ATHENA
10,07.01.2024,NaN,NaN
11,12:52,9398917,COSTA\nDELIZIOSA
12,10.01.2024,NaN,NaN
13,08:53,9432799,LA BELLE DE\nL'ADRIATIQUE
14,28.01.2024,NaN,NaN


## Data cleaning

In [38]:
df['arrival_date'] = pd.to_datetime(
    df['0'],
    format = '%d.%m.%Y' ,
    errors = 'coerce'
)

In [39]:
df['arrival_date'] = df['arrival_date'].ffill()

In [40]:
ships = df[
    df['0'].astype(str).str.fullmatch(r'\d{2}:\d{2}', na=False)].copy()
ships[['arrival_date', '0' ,'2']].head(10)

,arrival_date,0,2
5,2024-01-03,21:25,ATHENA
7,2024-01-04,08:12,COSTA\nDELIZIOSA
9,2024-01-05,00:13,ATHENA
11,2024-01-07,12:52,COSTA\nDELIZIOSA
13,2024-01-10,08:53,LA BELLE DE\nL'ADRIATIQUE
15,2024-01-28,22:01,ATHENA
21,2024-02-06,21:30,ATHENA
23,2024-02-07,21:42,ATHENA
25,2024-02-17,21:27,ATHENA
27,2024-02-20,23:30,ATHENA


In [41]:
ships.shape

(932, 15)

In [42]:
ships[['arrival_date', '0', '2']].isnull().sum()

arrival_date    0
0               0
2               0
dtype: int64

In [43]:
ships['arrival_date'].dt.year.value_counts().sort_index()

arrival_date
2024    520
2025    412
Name: count, dtype: int64

In [44]:
ships = ships[
    ((ships['arrival_date'] >= '2024-04-19') & 
     (ships['arrival_date'] < '2024-10-04')) |
    ((ships['arrival_date'] >= '2025-07-11') &
     (ships['arrival_date'] < '2025-08-30'))
    ].copy()

In [45]:
ships.shape

(487, 15)

## Feature engineering

In [48]:
ships['arrival_datetime'] = pd.to_datetime(
    ships['arrival_date'].dt.strftime('%Y-%m-%d') + ' ' + ships['0'])

In [49]:
ships['year'] = ships['arrival_datetime'].dt.year

In [52]:
ships = ships[['arrival_datetime', 'arrival_date', 'year']].copy()

## Daily cruise arrivals

In [55]:
cruise_daily = (
    ships.groupby('arrival_date')
    .size()
    .reset_index(name = 'cruise_count')
)
cruise_daily.head()

,arrival_date,cruise_count
0,2024-04-22,1
1,2024-04-23,1
2,2024-04-24,3
3,2024-04-25,1
4,2024-04-26,2


In [57]:
dates_2024 = pd.date_range('2024-04-19', '2024-10-03')
dates_2025 = pd.date_range('2025-07-11', '2025-08-29')

all_dates = dates_2024.append(dates_2025)
cruise_daily = (
    pd.DataFrame({'arrival_date' : all_dates})
    .merge(cruise_daily, on='arrival_date', how = 'left')
    .fillna({'cruise_count' : 0})
)
cruise_daily['cruise_count'] = cruise_daily['cruise_count'].astype(int)
cruise_daily.head(10)

,arrival_date,cruise_count
0,2024-04-19,0
1,2024-04-20,0
2,2024-04-21,0
3,2024-04-22,1
4,2024-04-23,1
5,2024-04-24,3
6,2024-04-25,1
7,2024-04-26,2
8,2024-04-27,1
9,2024-04-28,1


In [58]:
cruise_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   arrival_date  218 non-null    datetime64[ns]
 1   cruise_count  218 non-null    int64         
dtypes: datetime64[ns](1), int64(1)
memory usage: 3.5 KB


## Exporting cleaned dataset

In [59]:
cruise_daily.to_csv(
    '../../data/processed_data/cruise_daily_cleaned.csv',
    index = False
)

## Hourly cruise arrivals

In [62]:
cruise_hourly = (
    ships.groupby(ships['arrival_datetime'].dt.floor('h'))
    .size()
    .reset_index(name = 'cruise_count_hourly')
)
cruise_hourly.head(10)
    
    

,arrival_datetime,cruise_count_hourly
0,2024-04-22 20:00:00,1
1,2024-04-23 07:00:00,1
2,2024-04-24 06:00:00,1
3,2024-04-24 07:00:00,2
4,2024-04-25 08:00:00,1
5,2024-04-26 07:00:00,2
6,2024-04-27 07:00:00,1
7,2024-04-28 13:00:00,1
8,2024-04-30 07:00:00,1
9,2024-04-30 08:00:00,1


In [63]:
cruise_hourly['arrival_date'] = cruise_hourly['arrival_datetime'].dt.normalize()

cruise_hourly = cruise_hourly.merge(
    cruise_daily.rename(columns={'cruise_count' : 'cruise_count_daily'}),
    on = 'arrival_date',
    how = 'left'
)
cruise_hourly.head()

,arrival_datetime,cruise_count_hourly,arrival_date,cruise_count_daily
0,2024-04-22 20:00:00,1,2024-04-22,1
1,2024-04-23 07:00:00,1,2024-04-23,1
2,2024-04-24 06:00:00,1,2024-04-24,3
3,2024-04-24 07:00:00,2,2024-04-24,3
4,2024-04-25 08:00:00,1,2024-04-25,1


In [64]:
cruise_hourly = cruise_hourly.drop(columns=['arrival_date'])
cruise_hourly.head()

,arrival_datetime,cruise_count_hourly,cruise_count_daily
0,2024-04-22 20:00:00,1,1
1,2024-04-23 07:00:00,1,1
2,2024-04-24 06:00:00,1,3
3,2024-04-24 07:00:00,2,3
4,2024-04-25 08:00:00,1,1


In [65]:
cruise_hourly.to_csv('../../data/processed_data/cruise_hourly_cleaned.csv', index=False)